# Lesson 33 — Exercises

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm

## Exercise 1 — Sizing the encodings (10 pts)

Take the table $b = 0, 1, \dots, 12$ (13 breakpoints) with values $\cos b$, on an input $x \in [0, 12]$.

1. **Before running anything**, use the formulas of Reading §2.1 to predict the number of binaries and of auxiliary continuous variables that `m.piecewise` adds under `"incremental"`, `"log"` and `"disaggregated"`.
2. Check your predictions with `m.num_integer` and `m.num_continuous` (remember that $x$ and $y$ are 2 of the continuous variables).
3. A table with only **two** breakpoints adds no binary under any method. Explain why, and confirm it for one method.
4. Which encoding would you choose for a 257-breakpoint property table used in a model that you must export to `.nl`? Justify the choice in two sentences.

**Expected:** (binaries, aux) $= (11, 12)$, $(4, 13)$ and $(12, 24)$.

In [ ]:
# TODO
bp = np.arange(13.0)
vals = np.cos(bp)

sizes = {}   # method -> (binaries, aux continuous), e.g. sizes["log"] = (..., ...)

if sizes:
    assert sizes["incremental"] == (11, 12)
    assert sizes["log"] == (4, 13)
    assert sizes["disaggregated"] == (12, 24)

## Exercise 2 — A tariff and the domain rule (15 pts)

Three sites share a load of 210 MWh. Each site buys power on the same block tariff, with cumulative cost (k\$) at these consumption breakpoints (MWh):

```
breakpoints = [0, 40, 70, 100]
cost        = [0, 240, 510, 630]
```

The marginal price is 6, then 9, then 4 k\$/MWh. The last band is a bulk discount, so the cost is **nonconvex**.

1. Declare each site's load with `ub=120` (the grid connection limit) and call `m.piecewise`. Explain the `PiecewiseDomainError` you get and why `discopt` refuses instead of quietly restricting the load to 100.
2. Fix it the first way: set `ub=100`. Minimize total cost subject to $\sum_i L_i = 210$. Report the certified optimum and the loads.
3. Fix it the second way: the utility confirms the bulk band continues at 3 k\$/MWh up to 120 MWh. Extend the table and keep `ub=120`. Report the new optimum and explain, from the marginal prices, why it differs.
4. In the model from part 2, widen the bound to 120 *after* declaring the table, then call `solve()`. What happens, and at which step is the check made?
5. Confirm both optima by brute-force enumeration on a 0.5 MWh grid.

**Expected:** $z^\star = 1320$ (with `ub=100`) and $z^\star = 1260$ (extended table).

In [ ]:
# TODO
breakpoints = [0, 40, 70, 100]
cost = [0, 240, 510, 630]
demand = 210.0

z_ub100 = None     # certified optimum with ub=100
z_ub120 = None     # certified optimum with the extended table

if z_ub100 is not None:
    assert abs(z_ub100 - 1320.0) < 1e-4
if z_ub120 is not None:
    assert abs(z_ub120 - 1260.0) < 1e-4

## Exercise 3 — Outer versus approximate PWL (15 pts)

$$\min_{x \in [-3, 3],\; y \in [-1, 2]} \; \sin 3x + 0.1 x^2 + (y - 1)^2 \quad \text{s.t.} \quad x + y \ge 1.$$

1. Solve it with `m.solve()` and report status, objective, bound and time.
2. Build `T = dm.nonlinear_to_pwl(m, segments=s)` for $s = 4$ and $s = 8$. List `T.terms` and `T.skipped`, and `T.fully_linear`. Which subexpressions were replaced?
3. Solve each with `T.solve(time_limit=60)`. Report status, objective, bound, `gap_certified`, the number of refinement rounds (`r.solver_stats["nonlinear_to_pwl"]["rounds"]`) and the final number of breakpoints per term. Does starting from more segments save rounds?
4. Repeat with `mode="approximate"`. Report `bound` and `gap_certified`, and explain why they are what they are even though the objective is right.
5. Compare the three routes' times, and say when you would choose the outer-PWL route over `m.solve()`.

**Expected:** all routes return $z^\star \approx -0.758628$ at $x \approx 1.5366$, $y = 1$. The outer mode certifies, and the approximate mode has `bound=None`.

In [ ]:
# TODO
def build_ex3():
    m = do.Model("ex3")
    x = m.continuous("x", lb=-3, ub=3)
    y = m.continuous("y", lb=-1, ub=2)
    m.minimize(dm.sin(3 * x) + 0.1 * x**2 + (y - 1) ** 2)
    m.subject_to(x + y >= 1)
    return m, x, y

z_outer = None          # objective from the outer-mode PWL solve
outer_certified = None  # its gap_certified flag
approx_bound = "unset"  # r.bound from the approximate-mode solve

if z_outer is not None:
    assert abs(z_outer - (-0.758628)) < 1e-4 and outer_certified is True
if approx_bound != "unset":
    assert approx_bound is None

## Exercise 4 — A Colebrook grey box (15 pts)

Size one water pipe ($L = 500$ m, $Q = 0.04$ m³/s, commercial steel, roughness $4.5\times10^{-5}$ m, $\rho = 1000$, $\mu = 10^{-3}$) to minimize annualized cost

$$\min_{D \in [0.08, 0.5]} \; 1200\,L\,D^{1.5} + b\,Q\,\Delta p, \qquad \Delta p = f(D)\,\frac{8\rho L Q^2}{\pi^2 D^5} \le 1.5\times10^5 \text{ Pa},$$

with $b = 8000 \cdot 0.12 / 0.75 / 1000$ \$/yr per W. The Darcy friction factor $f$ solves the Colebrook equation {cite:p}`Colebrook1939`. With $s = 1/\sqrt f$, $k = 2/\ln 10$, $e = \varepsilon/3.7$ and $c = 2.51\pi\mu/(4\rho Q)$, it is the root of

$$G(s, D) = s + k \ln u = 0, \qquad u = e/D + cDs.$$

1. Write `friction(D)` that solves $G = 0$ by Newton's method (start at $s = 7$).
2. Derive $f'(D)$ and $f''(D)$ by the implicit function theorem, using $G_s = 1 + kcD/u$, $G_D = k u_D/u$ with $u_D = -e/D^2 + cs$, $G_{ss} = -k(cD)^2/u^2$, $G_{Ds} = kc(u - D u_D)/u^2$ and $G_{DD} = k(2eu/D^3 - u_D^2)/u^2$. Then $s' = -G_D/G_s$, $s'' = -(G_{DD} + 2G_{Ds}s' + G_{ss}s'^2)/G_s$, $f' = -2s^{-3}s'$ and $f'' = 6s^{-4}s'^2 - 2s^{-3}s''$.
3. Check $f'$ and $f''$ against central differences at $D = 0.1, 0.2, 0.3$. The relative error should be below $10^{-6}$.
4. Wrap the solver with `dm.external` on a **scalar** `D` (so `jac` and `hess` return shape `(1,)`), count calls, and solve. Report status, cost, $D^\star$ and bound.
5. Build the symbolic twin ($s$ a variable in $[2, 20]$ with $G(s, D) = 0$ as a constraint) and solve it. Compare, and state which of the two results is a certificate.

**Expected:** $D^\star \approx 0.14865$ m and cost $\approx 42066.33$ \$/yr from both. Only the twin reports `optimal` with `gap_certified=True`.

In [ ]:
# TODO
RHO, MU, ROUGH = 1000.0, 1.0e-3, 4.5e-5
L_PIPE, Q_FLOW = 500.0, 0.04
K = 2.0 / np.log(10.0)
E = ROUGH / 3.7
C = 2.51 * np.pi * MU / (4.0 * RHO * Q_FLOW)
A_CAP = 1200.0
B_POW = 8000 * 0.12 / 0.75 / 1000
G_DP = 8 * RHO * L_PIPE * Q_FLOW**2 / np.pi**2      # dp = f * G_DP / D**5

D_greybox = None     # optimal diameter from the dm.external model
cost_greybox = None
cost_twin = None     # certified cost from the symbolic twin

if D_greybox is not None:
    assert abs(D_greybox - 0.14865) < 1e-4 and abs(cost_greybox - 42066.33) < 0.05
if cost_twin is not None:
    assert abs(cost_twin - 42066.33) < 0.05

## Exercise 5 — A table *and* a grey box (15 pts)

Return to the reactor of Reading §5, but price the reactor from a vendor table. The annualized charge (per unit time) for residence time $\tau$ is tabulated, with a size class at each breakpoint:

```
tau_bp = [0.1, 1.0, 3.0, 6.0, 10.0]
vcost  = [1.0, 5.0, 11.0, 20.0, 45.0]
```

Maximize $100\,X(\tau, T) - \text{vcost}(\tau) - 0.15\,(T - 300)$ over $\tau \in [0.1, 10]$, $T \in [300, 400]$.

1. Build the model with the grey-box `cstr` block (copied below) and `m.piecewise` for the cost. Solve it, and explain the error you get in terms of what branch and bound needs at every node.
2. **Route A, certify.** Replace the grey box by its symbolic twin and solve. Report the status, profit, $(\tau, T)$ and bound.
3. **Route B, enumerate.** The cost is *linear* on each table segment. For each of the 4 segments, solve the grey-box NLP with $\tau$ restricted to that segment and the segment's linear cost, from two starting points. Take the best. Which segment wins?
4. Compare A and B. What exactly does each route prove? Would route B still be trustworthy with a real simulator that has no symbolic twin? What would you add to make it more convincing?

**Expected:** profit $\approx 43.749$ at $\tau \approx 3.8685$, $T = 400$, in segment 3 ($\tau \in [3, 6]$), from both routes. Only route A is `optimal`.

In [ ]:
# TODO
import logging
logging.getLogger("discopt").setLevel(logging.ERROR)

k0, EoR, CA0 = 4.0e6, 6000.0, 2.0

def cstr_solve(tau, T):
    k = k0 * np.exp(-EoR / T)
    a = k * tau * CA0
    X = 0.5
    for _ in range(100):
        step = (a * (1 - X) ** 2 - X) / (-2 * a * (1 - X) - 1)
        X -= step
        if abs(step) < 1e-15:
            break
    GX = -2 * a * (1 - X) - 1
    return X, -np.array([k * CA0 * (1 - X) ** 2, a * EoR / T**2 * (1 - X) ** 2]) / GX

def _hess(z):
    z = np.asarray(z, dtype=float)
    H = np.zeros((1, 2, 2))
    for j in range(2):
        e = np.zeros(2)
        e[j] = 1e-6 * max(1.0, abs(z[j]))
        H[0, :, j] = (cstr_solve(*(z + e))[1] - cstr_solve(*(z - e))[1]) / (2 * e[j])
    return 0.5 * (H + H.transpose(0, 2, 1))

cstr = dm.external(lambda z: np.array([cstr_solve(z[0], z[1])[0]]),
                   jac=lambda z: cstr_solve(z[0], z[1])[1][None, :],
                   hess=_hess, shape=(1,), name="cstr")

tau_bp = [0.1, 1.0, 3.0, 6.0, 10.0]
vcost = [1.0, 5.0, 11.0, 20.0, 45.0]

profit_A = None   # certified profit, route A
profit_B = None   # best profit over the segments, route B
tau_B = None

if profit_A is not None:
    assert abs(profit_A - 43.749) < 1e-3
if profit_B is not None:
    assert abs(profit_B - 43.749) < 1e-3 and abs(tau_B - 3.8685) < 1e-3